# ForceLorentz — `force_lorentz`

`Postraitement_fields` · ASN 28 validation form (TRUST-LiMe).



## Introduction

Validation of the **differential** post-treatment fields of the standalone
magnetic (induction) equation:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

This notebook runs **one and only one test case**: `force_lorentz`. Per the
TRUST validation-form layout the other `Postraitement_fields` cases live in
their own sibling folders.

## What it validates

$$
\mathbf{F} = \mathbf{J} \times \mathbf{b} = (\nabla \times \mathbf{b}) \times \mathbf{b} / \mu_0
$$
Exactly linear field $b = (-y, x)$: $w_z = 2$, so with $\mu_0 = 1$:
$F = (-w_z b_y,\ w_z b_x) = (-2x,\ -2y)$ in every cell. At the probed cell
centre $(0.45, 0.55)$: $F = (-0.9, -1.1)$. (In 3D the Lorentz force is a
full vector; here $b$ is planar so $w$ is out-of-plane and $F$ is in-plane.)


## Contract checked

| # | Quantity | Formula |
|---|---|---|
| 1 | Lorentz force density | $F = (\nabla\times b)\times b / \mu_0$ |

All these fields are built from the module-reconstructed element gradient
$\nabla b$ (`gradient_b_elements`, Green-Gauss — the kernel has no
element-field gradient factory on PolyMAC). Vector model required: the
scalar model is rejected with a clean error by `creer_champ`.



In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()



## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.



In [ ]:
import numpy as np
import os
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, sonde):
    """Return (time, values) for probe `sonde` of case `case`."""
    cands = [f"{BUILD}/coarse/{case}_{sonde}.son",
             f"{BUILD}/{case}_{sonde}.son"]
    if case.startswith("PAR_"):      # per-case subdir (merged MMS forms)
        cands.insert(1, f"{BUILD}/{case[4:]}/coarse/{case}_{sonde}.son")
    path = next((p for p in cands if os.path.exists(p)), cands[1])
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def read_son_all(case, sonde):
    """Like read_son but returns every entry (components of a vector field)."""
    cands = [f"{BUILD}/coarse/{case}_{sonde}.son",
             f"{BUILD}/{case}_{sonde}.son"]
    if case.startswith("PAR_"):      # per-case subdir (merged MMS forms)
        cands.insert(1, f"{BUILD}/{case[4:]}/coarse/{case}_{sonde}.son")
    path = next((p for p in cands if os.path.exists(p)), cands[1])
    son = SonFile(path, None)
    return [(e, np.asarray(son.getValues(e)[1], dtype=float)) for e in son.getEntries()]

print("BUILD =", BUILD)



## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).



In [ ]:
from trustutils import run

NB_PROCS = 2

run.reset()
# --- mesh-convergence scan (shared ladder) ----------------------------------
# The single dataset src/force_lorentz.data carries $nx/$ny node-count placeholders;
# every scale runs the SAME file via trustutils substitution
# (run.addCaseFromTemplate -> string.Template) into build/<scale>/ — one
# data-file name across scales:
#   coarse   = the baseline mesh
#   standard = cells x2 (h/2)
#   fine     = cells x4 (h/4)
# Each scale runs in its own build/<scale>/ directory so that the
# partitioned DOM_000*.Zones of one level never overwrite another's.
NX0 = NY0 = 11                       # baseline node counts (coarse)
meshes = {
    "coarse":   {"nx": 11, "ny": 11},            # the baseline mesh
    "standard": {"nx": 2*11-1, "ny": 2*11-1},
    "fine":     {"nx": 4*11-3, "ny": 4*11-3},
}
cases = {}
for lvl, dic in meshes.items():
    cases[lvl] = run.addCaseFromTemplate("force_lorentz.data", lvl, dic=dic,
                                         nbProcs=NB_PROCS, excluNR=True)
    cases[lvl].partition()          # trust -partition force_lorentz.data 2 -> PAR_force_lorentz.data + DOM_000*.Zones
run.printCases()
run.runCases()
case = cases["coarse"]            # alias kept for the read-back cells


## Checks


In [ ]:
entries = read_son_all(case.dataFileName_, "SONDE_FL")
for name, v in entries:
    print(name, "=", v[-1])
comps = [v[-1] for name, v in entries]

# b = (-y, x) is exactly linear => F = (-2 x, -2 y) with mu0 = 1.
expected = np.array([-2 * 0.45, -2 * 0.55])
print(f"force_lorentz (probe) : ({comps[0]:.6g}, {comps[1]:.6g})  (expected {expected})")
err = np.max(np.abs(np.array(comps) - expected)) / np.max(np.abs(expected))
print(f"relative error        : {err:.3e}")
assert err < 1e-12, f"force_lorentz wrong: {comps}"
print("OK: force_lorentz = (curl b) x b / mu0.")


## Mesh convergence

The shared refinement ladder: `coarse` = the baseline mesh above (`11x11`),
`standard` = cells x2 (`21x21`), `fine` = cells x4 (`41x41`).

The field is reconstructed from a linear/uniform `b` at an interior probe,
which is a **discrete identity** (the Green-Gauss reconstruction is exact
for linear fields at interior cells): b = (-y,x) linear => F = (curl b) x b / mu0 = (-2x, -2y), position-dependent. The scan asserts the same
machine-precision bound at **every** level
(evaluated at the probe's effective cell-centre position from the .son header) — no order is reported.

In [ ]:
def probe_pos(path):
    """Effective (x, y) of a point probe, from the .son header."""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("# Temps"):
                toks = line.replace("=", " = ").split()
                return (float(toks[toks.index("x") + 2]), float(toks[toks.index("y") + 2]))
    raise RuntimeError(f"no probe position header in {path}")
def read_final_all_dir(subdir, case_name, name):
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    return [np.asarray(son.getValues(e)[1], dtype=float) for e in son.getEntries()]

def level_error(subdir, case_name):
    xy = probe_pos(f"{BUILD}/{subdir}/{case_name}_SONDE_FL.son")
    comps = [float(vv[-1]) for vv in read_final_all_dir(subdir, case_name, "SONDE_FL")]
    expected = np.array([-2 * xy[0], -2 * xy[1]])
    e = float(np.max(np.abs(np.array(comps) - expected)))
    return e, expected

scan = {"coarse": ("coarse", case.dataFileName_),
        "standard": (f"standard", cases["standard"].dataFileName_),
        "fine": (f"fine", cases["fine"].dataFileName_)}
scan_nodes = {"coarse": (11, 11), "standard": (21, 21), "fine": (41, 41)}
res = {lvl: level_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes        abs err        rel")
for lvl in ("coarse", "standard", "fine"):
    n0, n1 = scan_nodes[lvl]
    e, expected = res[lvl]
    rel = e / float(np.max(np.abs(expected)))
    print(f"{lvl:<9} {n0:>3}x{n1:>3} {e:<13.3e} {rel:.1e}")
    assert rel < 1e-12, f"identity broken at {lvl}: {rel:.3e}"
print("OK: the reconstruction identity holds at every mesh level.")

## Computer Performance


In [ ]:
run.tablePerf()

